# Hậu xử lý `endocarp` theo hình dạng quả

Notebook này giữ phần core của hai nhánh:

- **Ớt dài**: top-cap hull cục bộ rồi fill phần miss bên trong.
- **Ớt chuông**: lấy trực tiếp mask `endocarp` thô từ YOLO, sau đó dùng connector làm biên trên và fill phần phía dưới; không qua bước hậu xử lý endocarp của pipeline.

Để file nhẹ, notebook chỉ giữ test tổng hợp ngắn và 4 crop ớt chuông đại diện. Gallery được lưu thành file ngoài notebook, không nhúng base64.

In [ ]:
from pathlib import Path
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src.detect import DetectModel
from src.object import Chili
from src.segment import SegmentModel

warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "runs" / "endocarp_shape_aware"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
detector = DetectModel(model_name="none")
segmenter = SegmentModel(
    weight_path=str(ROOT / "pretrain" / "yolov8_segment_v1.pt"),
    model_name="yolo",
    device=DEVICE,
)
print("Device:", DEVICE)

## Phương pháp

Phân loại bằng tỷ lệ bbox: `aspect_ratio >= 1.5` là ớt dài, còn lại là ớt chuông.

### Nhánh dài: `long_top_cap_hull_fill`

Giữ nguyên baseline `chili.mask_endocarp` đã qua pipeline và phương pháp top-cap hiện tại: nối hai đỉnh trên, tạo convex hull cục bộ chỉ ở phần đầu, sau đó flood-fill phần miss nằm bên trong.

### Nhánh chuông: `bell_inner_bridge_clip_fill`

1. Gộp trực tiếp các mask class `endocarp` (`class_id=3`) trong `chili.rseg['masks_data']`; không dùng polygon/hole-fill hậu xử lý của pipeline làm baseline.
2. Tạo `stem_guard` bằng cách giãn stem 2 px và tạo ROI `mask_chili - mask_outer - stem_guard`.
3. Lấy điểm trong cao nhất của nhánh trái/phải, nhưng điểm và đường nối 3 px phải nằm dưới guard.
4. Trong khoảng ngang giữa hai endpoint, xóa pixel baseline nằm phía trên connector; ngoài khoảng này giữ nguyên.
5. Tạo `closed_mask = clipped_baseline OR connector`, flood-fill vùng kín; sau đó fill khoảng giữa biên trái/phải theo từng hàng để lấp cả khe hở không kín. Mọi pixel bổ sung đều bị giới hạn phía dưới connector và trong ROI.

Không gọi `convexHull`/`fillConvexPoly` ở nhánh chuông. Trong khoảng hai endpoint, output không thể vượt lên trên biên connector.

In [ ]:
SHAPE_RATIO_THRESHOLD = 1.5
LINE_THICKNESS = 3
BELL_STEM_CLEARANCE_PX = 2


def as_binary(mask):
    return ((mask > 0).astype(np.uint8) * 255)


def component_count(mask):
    count, _ = cv2.connectedComponents((mask > 0).astype(np.uint8), connectivity=8)
    return max(0, count - 1)


def classify_shape(width_bbox, height_bbox, threshold=SHAPE_RATIO_THRESHOLD):
    short_side = max(1e-6, min(float(width_bbox), float(height_bbox)))
    aspect_ratio = max(float(width_bbox), float(height_bbox)) / short_side
    shape_type = "long" if aspect_ratio >= threshold else "bell"
    return shape_type, aspect_ratio


def overlay(image, mask, color, alpha=0.42):
    result = image.copy()
    active = mask > 0
    color_array = np.asarray(color, dtype=np.float32)
    result[active] = (
        result[active].astype(np.float32) * (1.0 - alpha)
        + color_array * alpha
    ).astype(np.uint8)
    return result


def build_top_fill_roi(chili_mask, outer_mask):
    return cv2.bitwise_and(as_binary(chili_mask), cv2.bitwise_not(as_binary(outer_mask)))


def find_top_peaks(mask, stem_mask, allowed_roi):
    candidate_mask = cv2.bitwise_and(as_binary(mask), as_binary(allowed_roi))
    empty = np.zeros_like(candidate_mask)
    ys, xs = np.where(candidate_mask > 0)
    if len(xs) == 0:
        return None, None, empty, {"reason": "empty_mask"}

    stem_xs = np.where(as_binary(stem_mask) > 0)[1]
    if len(stem_xs) > 0:
        center_x = int(round(np.median(stem_xs)))
        split_source = "stem_center"
    else:
        center_x = int(round((int(xs.min()) + int(xs.max())) / 2.0))
        split_source = "mask_bbox_center"

    contours, _ = cv2.findContours(
        candidate_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE
    )
    if not contours:
        return None, None, empty, {"reason": "empty_contour"}

    boundary = np.vstack([contour.reshape(-1, 2) for contour in contours])
    left_boundary = boundary[boundary[:, 0] < center_x]
    right_boundary = boundary[boundary[:, 0] > center_x]
    if len(left_boundary) == 0 or len(right_boundary) == 0:
        return None, None, empty, {
            "reason": "missing_side_boundary",
            "center_x": center_x,
            "split_source": split_source,
        }

    def highest_inner_point(points, side):
        top_y = int(points[:, 1].min())
        top_row = points[points[:, 1] == top_y]
        top_x = int(top_row[:, 0].max() if side == "left" else top_row[:, 0].min())
        return np.array([top_x, top_y], dtype=np.int32)

    left_point = highest_inner_point(left_boundary, "left")
    right_point = highest_inner_point(right_boundary, "right")
    if int(left_point[0]) >= int(right_point[0]):
        return None, None, empty, {
            "reason": "invalid_peak_order",
            "center_x": center_x,
            "split_source": split_source,
        }

    debug = {
        "reason": "peaks_found",
        "center_x": center_x,
        "split_source": split_source,
        "left_point": left_point.tolist(),
        "right_point": right_point.tolist(),
        "left_peak_y": int(left_point[1]),
        "right_peak_y": int(right_point[1]),
        "cut_y": max(int(left_point[1]), int(right_point[1])),
        "endpoints_valid": bool(
            candidate_mask[left_point[1], left_point[0]] > 0
            and candidate_mask[right_point[1], right_point[0]] > 0
        ),
    }
    return left_point, right_point, candidate_mask, debug


def build_local_top_cap(
    baseline, candidate_mask, left_point, right_point, allowed_roi,
    line_thickness=LINE_THICKNESS,
    skip_if_centerline_covered=True,
):
    empty = np.zeros_like(candidate_mask)
    cut_y = max(int(left_point[1]), int(right_point[1]))

    centerline = empty.copy()
    cv2.line(
        centerline, tuple(left_point), tuple(right_point), 255,
        thickness=1, lineType=cv2.LINE_8,
    )
    if np.any((centerline > 0) & (allowed_roi == 0)):
        return empty, empty, empty, {"reason": "connector_outside_roi", "cut_y": cut_y}
    centerline_gap = cv2.bitwise_and(centerline, cv2.bitwise_not(as_binary(baseline)))
    if skip_if_centerline_covered and np.count_nonzero(centerline_gap) == 0:
        return empty, empty, empty, {"reason": "ok", "cut_y": cut_y, "connector_needed": False}

    bridge = empty.copy()
    fallback_fill = empty.copy()
    cv2.line(
        bridge, tuple(left_point), tuple(right_point), 255,
        thickness=line_thickness, lineType=cv2.LINE_8,
    )
    bridge = cv2.bitwise_and(bridge, allowed_roi)
    bridge[cut_y + 1:] = 0

    top_mask = candidate_mask.copy()
    top_mask[cut_y + 1:] = 0
    count, labels = cv2.connectedComponents(
        (top_mask > 0).astype(np.uint8), connectivity=8
    )
    left_label = int(labels[left_point[1], left_point[0]])
    right_label = int(labels[right_point[1], right_point[0]])
    if count <= 1 or left_label == 0 or right_label == 0:
        return empty, empty, bridge, {"reason": "invalid_top_components", "cut_y": cut_y}

    selected_top = np.where(
        (labels == left_label) | (labels == right_label), 255, 0
    ).astype(np.uint8)
    cap_seed = cv2.bitwise_or(selected_top, bridge)
    seed_ys, seed_xs = np.where(cap_seed > 0)
    if len(seed_xs) < 3:
        return empty, empty, bridge, {"reason": "insufficient_hull_points", "cut_y": cut_y}

    points = np.column_stack((seed_xs, seed_ys)).astype(np.int32).reshape(-1, 1, 2)
    hull = cv2.convexHull(points)
    if hull is None or len(hull) < 3:
        return empty, empty, bridge, {"reason": "degenerate_hull", "cut_y": cut_y}

    top_hull = empty.copy()
    cv2.fillConvexPoly(top_hull, hull, 255)
    top_hull[cut_y + 1:] = 0
    top_hull = cv2.bitwise_and(top_hull, allowed_roi)
    local_fill = cv2.bitwise_and(top_hull, cv2.bitwise_not(as_binary(baseline)))
    return top_hull, local_fill, bridge, {"reason": "ok", "cut_y": cut_y, "connector_needed": True}


def fill_enclosed_interior(mask, allowed_roi):
    binary = as_binary(mask)
    padded = cv2.copyMakeBorder(binary, 1, 1, 1, 1, cv2.BORDER_CONSTANT, value=0)
    flooded = padded.copy()
    flood_mask = np.zeros(
        (padded.shape[0] + 2, padded.shape[1] + 2), dtype=np.uint8
    )
    cv2.floodFill(flooded, flood_mask, (0, 0), 255)
    enclosed = cv2.bitwise_not(flooded)[1:-1, 1:-1]
    enclosed[binary > 0] = 0
    return cv2.bitwise_and(enclosed, as_binary(allowed_roi))


def keep_additions_attached_to_baseline(additions, baseline):
    additions = as_binary(additions)
    baseline_contact = cv2.dilate(
        as_binary(baseline), np.ones((3, 3), dtype=np.uint8)
    )
    count, labels = cv2.connectedComponents(
        (additions > 0).astype(np.uint8), connectivity=8
    )
    kept = np.zeros_like(additions)
    for component_id in range(1, count):
        component = labels == component_id
        if np.any(component & (baseline_contact > 0)):
            kept[component] = 255
    return kept


def long_endocarp_postprocess(baseline, chili_mask, outer_mask, stem_mask):
    baseline = as_binary(baseline)
    stem_mask = as_binary(stem_mask)
    allowed_roi = build_top_fill_roi(chili_mask, outer_mask)
    components_before = component_count(baseline)
    empty = np.zeros_like(baseline)

    left, right, candidate_mask, peak_debug = find_top_peaks(
        baseline, stem_mask, allowed_roi
    )
    top_hull = empty.copy()
    top_fill = empty.copy()
    interior_fill = empty.copy()
    local_fill = empty.copy()
    bridge = empty.copy()
    bridge_debug = dict(peak_debug)

    if peak_debug.get("reason") == "peaks_found":
        top_hull, top_fill, bridge, cap_debug = build_local_top_cap(
            baseline, candidate_mask, left, right, allowed_roi
        )
        bridge_debug.update(cap_debug)
        bridge_debug["left_point"] = left.tolist()
        bridge_debug["right_point"] = right.tolist()
        bridge_debug["left_peak_y"] = int(left[1])
        bridge_debug["right_peak_y"] = int(right[1])
        bridge_debug["endpoints_valid"] = bool(peak_debug["endpoints_valid"])
        if cap_debug.get("reason") == "ok":
            closed_mask = cv2.bitwise_or(baseline, top_hull)
            interior_fill = fill_enclosed_interior(closed_mask, allowed_roi)
            local_fill = cv2.bitwise_or(top_fill, interior_fill)

    if bridge_debug.get("reason") == "ok":
        final = cv2.bitwise_or(baseline, local_fill)
        status = "top_cap_filled" if np.count_nonzero(local_fill) else "already_top_closed"
    else:
        final = baseline.copy()
        status = "unresolved"

    cut_y = bridge_debug.get("cut_y")
    below_cut_xor_px = 0
    if cut_y is not None and int(cut_y) + 1 < baseline.shape[0]:
        below_cut_xor_px = int(np.count_nonzero(
            cv2.bitwise_xor(baseline[int(cut_y) + 1:], final[int(cut_y) + 1:])
        ))
    removed_px = int(np.count_nonzero((baseline > 0) & (final == 0)))
    added_outside_roi_px = int(np.count_nonzero(
        (baseline == 0) & (final > 0) & (allowed_roi == 0)
    ))

    return {
        "branch": "long_top_cap_hull_fill",
        "status": status,
        "baseline": baseline,
        "final": final,
        "allowed_roi": allowed_roi,
        "stem_mask": stem_mask,
        "candidate_mask": candidate_mask,
        "top_hull": top_hull,
        "top_fill": top_fill,
        "interior_fill": interior_fill,
        "bridge": bridge,
        "local_fill": local_fill,
        "bridge_debug": bridge_debug,
        "components_before": components_before,
        "components_after": component_count(final),
        "removed_px": removed_px,
        "added_outside_roi_px": added_outside_roi_px,
        "below_cut_xor_px": below_cut_xor_px,
        "stem_overlap_px": int(np.count_nonzero((final > 0) & (stem_mask > 0))),
        "endpoints_valid": bridge_debug.get("endpoints_valid"),
        "xor_px": int(np.count_nonzero(cv2.bitwise_xor(baseline, final))),
    }


def build_bell_stem_guard(stem_mask, clearance_px=BELL_STEM_CLEARANCE_PX):
    stem_mask = as_binary(stem_mask)
    if not np.any(stem_mask):
        return np.zeros_like(stem_mask)
    kernel_size = 2 * int(clearance_px) + 1
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (kernel_size, kernel_size)
    )
    return cv2.dilate(stem_mask, kernel)


def find_bell_inner_peaks(
    mask, stem_guard, allowed_roi, line_thickness=LINE_THICKNESS
):
    candidate_mask = cv2.bitwise_and(as_binary(mask), as_binary(allowed_roi))
    empty = np.zeros_like(candidate_mask)
    guard_ys, guard_xs = np.where(stem_guard > 0)
    if len(guard_xs) == 0:
        return None, None, candidate_mask, {"reason": "no_stem"}

    stem_left = int(guard_xs.min())
    stem_right = int(guard_xs.max())
    stem_bottom = int(guard_ys.max())
    half_line = (int(line_thickness) + 1) // 2
    first_y = stem_bottom + half_line + 1

    ys, xs = np.where(candidate_mask > 0)
    left_selector = (xs < stem_left) & (ys >= first_y)
    right_selector = (xs > stem_right) & (ys >= first_y)
    if not np.any(left_selector) or not np.any(right_selector):
        return None, None, candidate_mask, {
            "reason": "missing_side_branch",
            "stem_left": stem_left, "stem_right": stem_right,
            "stem_bottom": stem_bottom, "first_y": first_y,
        }

    def highest_inner_pixel(side_xs, side_ys, side):
        top_y = int(side_ys.min())
        top_xs = side_xs[side_ys == top_y]
        top_x = int(top_xs.max() if side == "left" else top_xs.min())
        return np.array([top_x, top_y], dtype=np.int32)

    left_point = highest_inner_pixel(xs[left_selector], ys[left_selector], "left")
    right_point = highest_inner_pixel(xs[right_selector], ys[right_selector], "right")
    if int(left_point[0]) >= int(right_point[0]):
        return None, None, candidate_mask, {"reason": "invalid_peak_order"}

    centerline = empty.copy()
    cv2.line(centerline, tuple(left_point), tuple(right_point), 255, 1, cv2.LINE_8)
    raw_bridge = empty.copy()
    cv2.line(
        raw_bridge, tuple(left_point), tuple(right_point), 255,
        thickness=line_thickness, lineType=cv2.LINE_8,
    )
    if np.any((centerline > 0) & (allowed_roi == 0)):
        reason = "connector_outside_roi"
    elif np.any((raw_bridge > 0) & (allowed_roi == 0)):
        reason = "thick_connector_outside_roi"
    elif np.any((raw_bridge > 0) & (stem_guard > 0)):
        reason = "connector_touches_stem"
    else:
        reason = "peaks_found"

    debug = {
        "reason": reason,
        "stem_left": stem_left, "stem_right": stem_right,
        "stem_bottom": stem_bottom, "first_y": first_y,
        "left_point": left_point.tolist(),
        "right_point": right_point.tolist(),
        "left_peak_y": int(left_point[1]),
        "right_peak_y": int(right_point[1]),
        "cut_y": max(int(left_point[1]), int(right_point[1])),
        "endpoints_valid": bool(
            candidate_mask[left_point[1], left_point[0]] > 0
            and candidate_mask[right_point[1], right_point[0]] > 0
        ),
    }
    return left_point, right_point, candidate_mask, debug


def build_bell_connector_regions(shape, left_point, right_point):
    above = np.zeros(shape, dtype=np.uint8)
    below = np.zeros(shape, dtype=np.uint8)
    left_x, left_y = map(int, left_point)
    right_x, right_y = map(int, right_point)
    if right_x <= left_x:
        return above, below

    half_line = (LINE_THICKNESS + 1) // 2
    for x_coord in range(left_x, right_x + 1):
        ratio = (x_coord - left_x) / float(right_x - left_x)
        line_y = left_y + ratio * (right_y - left_y)
        top_edge_y = max(0, int(np.floor(line_y)) - half_line)
        above[:top_edge_y, x_coord] = 255
        below[top_edge_y:, x_coord] = 255
    return above, below


def fill_between_bell_sides(mask, allowed_roi, below_connector):
    row_spans = np.zeros_like(mask)
    source = as_binary(mask)
    for y_coord in range(source.shape[0]):
        row_xs = np.where(source[y_coord] > 0)[0]
        if len(row_xs) >= 2:
            row_spans[y_coord, int(row_xs.min()):int(row_xs.max()) + 1] = 255
    row_spans = cv2.bitwise_and(row_spans, as_binary(allowed_roi))
    row_spans = cv2.bitwise_and(row_spans, as_binary(below_connector))
    return cv2.bitwise_and(row_spans, cv2.bitwise_not(source))


def bell_endocarp_postprocess(baseline, chili_mask, outer_mask, stem_mask):
    baseline = as_binary(baseline)
    stem_mask = as_binary(stem_mask)
    stem_guard = build_bell_stem_guard(stem_mask)
    allowed_roi = build_top_fill_roi(chili_mask, outer_mask)
    allowed_roi[stem_guard > 0] = 0
    empty = np.zeros_like(baseline)

    left, right, candidate_mask, peak_debug = find_bell_inner_peaks(
        baseline, stem_guard, allowed_roi
    )
    top_hull = empty.copy()
    top_fill = empty.copy()
    interior_fill = empty.copy()
    between_sides_fill = empty.copy()
    local_fill = empty.copy()
    clipped_baseline = baseline.copy()
    above_connector = empty.copy()
    below_connector = empty.copy()
    removed_above_connector = empty.copy()
    discarded_island_px = 0
    bridge = empty.copy()
    bridge_debug = dict(peak_debug)

    if peak_debug.get("reason") == "peaks_found":
        cv2.line(
            bridge, tuple(left), tuple(right), 255,
            thickness=LINE_THICKNESS, lineType=cv2.LINE_8,
        )
        bridge = cv2.bitwise_and(bridge, allowed_roi)
        bridge_debug["reason"] = "ok"
        bridge_debug["connector_needed"] = bool(np.any(
            (bridge > 0) & (baseline == 0)
        ))
        bridge_debug["left_point"] = left.tolist()
        bridge_debug["right_point"] = right.tolist()
        bridge_debug["left_peak_y"] = int(left[1])
        bridge_debug["right_peak_y"] = int(right[1])
        bridge_debug["endpoints_valid"] = bool(peak_debug["endpoints_valid"])
        if bridge_debug.get("reason") == "ok":
            above_connector, below_connector = build_bell_connector_regions(
                baseline.shape, left, right
            )
            removed_above_connector = cv2.bitwise_and(
                baseline, above_connector
            )
            clipped_baseline = baseline.copy()
            clipped_baseline[above_connector > 0] = 0
            closed_mask = cv2.bitwise_or(clipped_baseline, bridge)
            interior_fill = fill_enclosed_interior(closed_mask, allowed_roi)
            interior_fill = cv2.bitwise_and(interior_fill, below_connector)
            between_sides_fill = fill_between_bell_sides(
                closed_mask, allowed_roi, below_connector
            )
            between_sides_fill = cv2.bitwise_and(
                between_sides_fill, cv2.bitwise_not(interior_fill)
            )
            raw_local_fill = cv2.bitwise_or(bridge, interior_fill)
            raw_local_fill = cv2.bitwise_or(raw_local_fill, between_sides_fill)
            raw_local_fill = cv2.bitwise_and(
                raw_local_fill, cv2.bitwise_not(clipped_baseline)
            )
            local_fill = keep_additions_attached_to_baseline(
                raw_local_fill, clipped_baseline
            )
            discarded_island_px = int(
                np.count_nonzero(raw_local_fill) - np.count_nonzero(local_fill)
            )
            interior_fill = cv2.bitwise_and(interior_fill, local_fill)
            between_sides_fill = cv2.bitwise_and(
                between_sides_fill, local_fill
            )

    reason = bridge_debug.get("reason")
    if reason == "ok":
        final = cv2.bitwise_or(clipped_baseline, local_fill)
        changed_px = np.count_nonzero(cv2.bitwise_xor(baseline, final))
        status = "bell_clipped_and_filled" if changed_px else "bell_already_closed"
    else:
        final = baseline.copy()
        # ── fallback fill ─────────────────────────────────────────────────
        # When find_bell_inner_peaks fails (no valid bridge is available),
        # the interior cavity between the two arm blobs would remain
        # completely unfilled. As a last resort, apply fill_between_bell_sides
        # using only allowed_roi (no below_connector restriction since there
        # is no reference bridge to anchor it). For every row that has pixels
        # on both sides the horizontal gap is filled.
        fallback_fill = fill_between_bell_sides(
            cv2.bitwise_and(baseline, allowed_roi),
            allowed_roi,
            np.full_like(allowed_roi, 255),
        )
        if np.any(fallback_fill > 0):
            final = cv2.bitwise_or(baseline, fallback_fill)
            status = "bell_fallback_filled"
        elif reason == "no_stem":
            status = "bell_unresolved_no_stem"
        elif reason == "missing_side_branch":
            status = "bell_unresolved_missing_branch"
        else:
            status = "bell_unresolved_invalid_connector"

    cut_y = bridge_debug.get("cut_y")
    below_cut_xor_px = 0
    if cut_y is not None and int(cut_y) + 1 < baseline.shape[0]:
        below_cut_xor_px = int(np.count_nonzero(
            cv2.bitwise_xor(baseline[int(cut_y) + 1:], final[int(cut_y) + 1:])
        ))

    return {
        "branch": "bell_inner_bridge_clip_fill",
        "status": status,
        "baseline": baseline, "final": final,
        "allowed_roi": allowed_roi,
        "stem_mask": stem_mask, "stem_guard": stem_guard,
        "candidate_mask": candidate_mask,
        "clipped_baseline": clipped_baseline,
        "above_connector": above_connector,
        "below_connector": below_connector,
        "removed_above_connector": removed_above_connector,
        "top_hull": top_hull, "top_fill": top_fill,
        "interior_fill": interior_fill,
        "between_sides_fill": between_sides_fill,
        "fallback_fill": fallback_fill,
        "local_fill": local_fill,
        "bridge": bridge, "bridge_debug": bridge_debug,
        "components_before": component_count(baseline),
        "components_after": component_count(final),
        "removed_px": int(np.count_nonzero((baseline > 0) & (final == 0))),
        "removed_above_connector_px": int(np.count_nonzero(removed_above_connector)),
        "final_above_connector_px": int(np.count_nonzero(
            (final > 0) & (above_connector > 0) & (bridge == 0)
        )),
        "added_outside_roi_px": int(np.count_nonzero(
            (baseline == 0) & (final > 0) & (allowed_roi == 0)
        )),
        "below_cut_xor_px": below_cut_xor_px,
        "stem_overlap_px": int(np.count_nonzero((final > 0) & (stem_guard > 0))),
        "connector_stem_overlap_px": int(np.count_nonzero((bridge > 0) & (stem_guard > 0))),
        "discarded_island_px": discarded_island_px,
        "endpoints_valid": bridge_debug.get("endpoints_valid"),
        "xor_px": int(np.count_nonzero(cv2.bitwise_xor(baseline, final))),
    }


def get_raw_yolo_class_mask(chili, class_id):
    raw_mask = np.zeros(chili.img.shape[:2], dtype=np.uint8)
    for mask_info in chili.rseg.get("masks_data", []):
        if (
            int(mask_info.get("class_id", -1)) == int(class_id)
            and mask_info.get("source", "yolo") == "yolo"
        ):
            single_mask = as_binary(mask_info["mask"])
            if single_mask.shape != raw_mask.shape:
                single_mask = cv2.resize(
                    single_mask, (raw_mask.shape[1], raw_mask.shape[0]),
                    interpolation=cv2.INTER_NEAREST,
                )
            raw_mask = cv2.bitwise_or(raw_mask, single_mask)
    return raw_mask


def shape_aware_endocarp_postprocess(chili):
    shape_type, aspect_ratio = classify_shape(chili.width_bbox, chili.height_bbox)
    if shape_type == "long":
        baseline = as_binary(chili.mask_endocarp)
        result = long_endocarp_postprocess(
            baseline, chili.mask_chili, chili.mask_outer, chili.mask_stem
        )
        result["baseline_source"] = "pipeline_postprocessed"
    else:
        baseline = get_raw_yolo_class_mask(chili, class_id=3)
        result = bell_endocarp_postprocess(
            baseline, chili.mask_chili, chili.mask_outer, chili.mask_stem
        )
        result["baseline_source"] = "raw_yolo_endocarp"
    result["shape_type"] = shape_type
    result["aspect_ratio"] = aspect_ratio
    return result


## Test tổng hợp ngắn

Kiểm tra ngưỡng phân loại, invariant của nhánh dài và các case connector/fill/unresolved của nhánh chuông. Test chuông xác nhận `top_hull = top_fill = 0`.

In [ ]:
def run_synthetic_tests():
    assert classify_shape(150, 100)[0] == "long"
    assert classify_shape(149, 100)[0] == "bell"

    fake_chili = type("FakeChili", (), {})()
    fake_chili.img = np.zeros((12, 16, 3), dtype=np.uint8)
    raw_a = np.zeros((12, 16), dtype=np.uint8)
    raw_b = np.zeros((12, 16), dtype=np.uint8)
    raw_a[2:5, 1:4] = 255
    raw_b[7:10, 10:14] = 255
    fake_chili.rseg = {"masks_data": [
        {"class_id": 3, "mask": raw_a, "source": "yolo"},
        {"class_id": 3, "mask": raw_b, "source": "yolo"},
        {"class_id": 2, "mask": np.full((12, 16), 255, np.uint8), "source": "yolo"},
    ]}
    assert np.array_equal(
        get_raw_yolo_class_mask(fake_chili, 3), cv2.bitwise_or(raw_a, raw_b)
    )

    shape = (100, 140)
    empty = np.zeros(shape, dtype=np.uint8)
    full_chili = np.full(shape, 255, dtype=np.uint8)
    stem = empty.copy()
    stem[2:35, 62:78] = 255

    closed = empty.copy()
    closed[25:80, 20:120] = 255
    closed_result = long_endocarp_postprocess(closed, full_chili, empty, stem)
    assert closed_result["status"] == "already_top_closed"
    assert closed_result["components_after"] == 1
    assert closed_result["removed_px"] == 0
    assert closed_result["below_cut_xor_px"] == 0

    asymmetric = empty.copy()
    asymmetric[12:80, 18:34] = 255
    asymmetric[30:80, 105:121] = 255
    asymmetric[68:80, 18:121] = 255
    bridge_result = long_endocarp_postprocess(asymmetric, full_chili, empty, stem)
    assert bridge_result["status"] == "top_cap_filled"
    assert bridge_result["components_after"] == 1
    assert bridge_result["bridge_debug"]["left_peak_y"] == 12
    assert bridge_result["bridge_debug"]["right_peak_y"] == 30
    assert bridge_result["bridge_debug"]["cut_y"] == 30
    assert bridge_result["removed_px"] == 0
    assert bridge_result["below_cut_xor_px"] > 0
    assert bridge_result["added_outside_roi_px"] == 0
    assert np.count_nonzero(bridge_result["interior_fill"]) > 0
    assert bridge_result["stem_overlap_px"] > 0
    assert bridge_result["endpoints_valid"] is True

    split = empty.copy()
    split[25:80, 10:35] = 255
    split[25:80, 105:130] = 255
    blocking_outer = empty.copy()
    blocking_outer[:, 60:80] = 255
    unresolved = long_endocarp_postprocess(
        split, full_chili, blocking_outer, stem
    )
    assert unresolved["status"] == "unresolved"
    assert unresolved["components_after"] == 2
    assert unresolved["xor_px"] == 0
    assert unresolved["bridge_debug"]["reason"] == "connector_outside_roi"

    bell_stem = empty.copy()
    bell_stem[2:25, 55:85] = 255
    bell_open = empty.copy()
    bell_open[30:85, 15:45] = 255
    bell_open[38:85, 95:125] = 255
    bell_open[72:85, 15:125] = 255
    bell_open[8:48, 55:85] = 255  # core/baseline sai nằm phía trên connector
    bell_result = bell_endocarp_postprocess(
        bell_open, full_chili, empty, bell_stem
    )
    bell_debug = bell_result["bridge_debug"]
    assert bell_result["status"] == "bell_clipped_and_filled"
    assert bell_debug["left_peak_y"] == 30
    assert bell_debug["right_peak_y"] == 38
    assert bell_debug["left_point"][0] < bell_debug["stem_left"]
    assert bell_debug["right_point"][0] > bell_debug["stem_right"]
    assert bell_result["connector_stem_overlap_px"] == 0
    assert bell_result["added_outside_roi_px"] == 0
    assert bell_result["removed_px"] > 0
    assert bell_result["removed_above_connector_px"] > 0
    assert bell_result["final_above_connector_px"] == 0
    assert bell_result["components_after"] <= bell_result["components_before"]
    assert np.count_nonzero(bell_result["interior_fill"]) > 0
    assert np.count_nonzero(bell_result["top_hull"]) == 0
    assert np.count_nonzero(bell_result["top_fill"]) == 0

    bell_closed = empty.copy()
    bell_closed[27:85, 15:125] = 255
    bell_closed_result = bell_endocarp_postprocess(
        bell_closed, full_chili, empty, bell_stem
    )
    assert bell_closed_result["status"] == "bell_already_closed"
    assert bell_closed_result["xor_px"] == 0

    bell_no_stem = bell_endocarp_postprocess(
        bell_open, full_chili, empty, empty
    )
    assert bell_no_stem["status"] == "bell_unresolved_no_stem"
    assert bell_no_stem["xor_px"] == 0

    bell_left_only = empty.copy()
    bell_left_only[30:85, 15:45] = 255
    bell_missing = bell_endocarp_postprocess(
        bell_left_only, full_chili, empty, bell_stem
    )
    assert bell_missing["status"] == "bell_unresolved_missing_branch"
    assert bell_missing["xor_px"] == 0
    print("Synthetic tests passed: long top cap; bell connector + enclosed fill (no hull)")


run_synthetic_tests()


## Test 4 crop ớt chuông đại diện

Dùng hai crop của `029_IT250232_2.jpg` (baseline đã kín) và hai crop của `030_IT264053_3.jpg` (cần connector/fill). Kết quả được ghi ra ngoài notebook để tránh làm file `.ipynb` nặng.

In [ ]:
BELL_SAMPLE_PATHS = [
    DATA_DIR / "029_IT250232_2.jpg",
    DATA_DIR / "030_IT264053_3.jpg",
]
for sample_path in BELL_SAMPLE_PATHS:
    if not sample_path.is_file():
        raise FileNotFoundError(sample_path)


def build_chili(source_path, bbox, candidate_index):
    # Chili hiện có thể cập nhật JSON annotation; khôi phục byte gốc sau inference.
    annotation_path = source_path.with_suffix(".json")
    snapshot = annotation_path.read_bytes() if annotation_path.is_file() else None
    try:
        return Chili(str(source_path), candidate_index, segmenter, bbox, ratio=1.0)
    finally:
        if snapshot is not None:
            current = annotation_path.read_bytes() if annotation_path.is_file() else None
            if current != snapshot:
                annotation_path.write_bytes(snapshot)


def collect_bell_samples(source_paths):
    samples, scan_rows = [], []
    for source_path in source_paths:
        try:
            boxes = detector.predict(str(source_path)) or []
        except Exception as exc:
            scan_rows.append({
                "source": source_path.name, "crop_index": None,
                "scan_status": "detection_error", "error": str(exc),
            })
            continue

        for crop_index, bbox in enumerate(boxes):
            values = [int(value) for value in bbox]
            shape_type, aspect_ratio = classify_shape(values[2], values[3])
            row = {
                "source": source_path.name, "crop_index": crop_index,
                "bbox_x": values[0], "bbox_y": values[1],
                "bbox_w": values[2], "bbox_h": values[3],
                "shape_type": shape_type, "aspect_ratio": round(aspect_ratio, 4),
                "scan_status": "excluded_long" if shape_type == "long" else "pending",
                "error": None,
            }
            if shape_type == "bell":
                try:
                    chili = build_chili(source_path, bbox, crop_index)
                    result = shape_aware_endocarp_postprocess(chili)
                    if result["shape_type"] != "bell":
                        raise AssertionError("BBox đổi nhánh sau khi tạo crop")
                    row["scan_status"] = "tested_bell"
                    samples.append({
                        "source_path": source_path, "crop_index": crop_index,
                        "bbox": values, "chili": chili, "result": result,
                    })
                except Exception as exc:
                    row["scan_status"] = "crop_error"
                    row["error"] = str(exc)
            scan_rows.append(row)
    return samples, pd.DataFrame(scan_rows)


bell_samples, data_scan_df = collect_bell_samples(BELL_SAMPLE_PATHS)
if not bell_samples:
    raise RuntimeError("Không tìm thấy crop ớt chuông trong data.")

rows = []
figure, axes = plt.subplots(
    len(bell_samples), 4, figsize=(18, 4.6 * len(bell_samples)), squeeze=False
)
for row_index, sample in enumerate(bell_samples):
    chili, result = sample["chili"], sample["result"]
    image, baseline, final = chili.img, result["baseline"], result["final"]
    debug = result["bridge_debug"]

    input_view = overlay(image, result["stem_guard"], (255, 0, 0), 0.42)
    baseline_view = overlay(image, baseline, (180, 0, 255), 0.46)
    debug_view = overlay(image, result["candidate_mask"], (0, 200, 220), 0.25)
    debug_view = overlay(debug_view, result["stem_guard"], (255, 0, 0), 0.42)
    debug_view = overlay(debug_view, result["removed_above_connector"], (255, 40, 40), 0.75)
    debug_view = overlay(debug_view, result["interior_fill"], (255, 120, 0), 0.58)
    debug_view = overlay(debug_view, result["between_sides_fill"], (0, 255, 80), 0.62)
    debug_view = overlay(debug_view, result["bridge"], (255, 255, 0), 1.0)
    if debug.get("left_point") is not None:
        cv2.circle(debug_view, tuple(debug["left_point"]), 6, (255, 255, 255), -1)
        cv2.circle(debug_view, tuple(debug["right_point"]), 6, (255, 255, 255), -1)
    final_view = overlay(image, final, (0, 220, 60), 0.46)

    label = f"{sample['source_path'].name} | crop_{sample['crop_index']}"
    views = [input_view, baseline_view, debug_view, final_view]
    titles = [
        f"{label}\naspect={result['aspect_ratio']:.3f} | stem guard",
        f"Raw YOLO endocarp\narea={np.count_nonzero(baseline):,}",
        f"Clip above + fill to connector | {result['status']}\n"
        f"removed={result['removed_px']:,} | enclosed={np.count_nonzero(result['interior_fill']):,} | gaps={np.count_nonzero(result['between_sides_fill']):,}",
        f"Final\narea={np.count_nonzero(final):,}",
    ]
    for axis, view, title in zip(axes[row_index], views, titles):
        axis.imshow(view)
        axis.set_title(title)
        axis.axis("off")

    left, right = debug.get("left_point"), debug.get("right_point")
    rows.append({
        "source": sample["source_path"].name, "crop_index": sample["crop_index"],
        "bbox_x": sample["bbox"][0], "bbox_y": sample["bbox"][1],
        "bbox_w": sample["bbox"][2], "bbox_h": sample["bbox"][3],
        "aspect_ratio": round(result["aspect_ratio"], 4),
        "baseline_source": result["baseline_source"],
        "branch": result["branch"], "status": result["status"],
        "bridge_status": debug.get("reason"),
        "components_before": result["components_before"],
        "components_after": result["components_after"],
        "baseline_area_px": int(np.count_nonzero(baseline)),
        "final_area_px": int(np.count_nonzero(final)),
        "added_px": int(np.count_nonzero((baseline == 0) & (final > 0))),
        "connector_px": int(np.count_nonzero(result["bridge"])),
        "top_hull_px": int(np.count_nonzero(result["top_hull"])),
        "top_fill_px": int(np.count_nonzero(result["top_fill"])),
        "interior_fill_px": int(np.count_nonzero(result["interior_fill"])),
        "between_sides_fill_px": int(np.count_nonzero(result["between_sides_fill"])),
        "removed_px": result["removed_px"],
        "removed_above_connector_px": result["removed_above_connector_px"],
        "final_above_connector_px": result["final_above_connector_px"],
        "added_outside_roi_px": result["added_outside_roi_px"],
        "connector_stem_overlap_px": result["connector_stem_overlap_px"],
        "stem_overlap_px": result["stem_overlap_px"],
        "stem_left": debug.get("stem_left"), "stem_right": debug.get("stem_right"),
        "stem_bottom": debug.get("stem_bottom"),
        "left_peak_x": left[0] if left is not None else None,
        "left_peak_y": left[1] if left is not None else None,
        "right_peak_x": right[0] if right is not None else None,
        "right_peak_y": right[1] if right is not None else None,
        "endpoints_valid": result["endpoints_valid"],
    })

figure.suptitle("4 mẫu ớt chuông: cắt phía trên + fill tới connector (không hull)", fontsize=16)
figure.tight_layout(rect=(0, 0, 1, 0.99))
figure_path = OUTPUT_DIR / "bell_sample_comparison.png"
figure.savefig(figure_path, dpi=130, bbox_inches="tight")
plt.close(figure)

bell_metrics_df = pd.DataFrame(rows)
metrics_path = OUTPUT_DIR / "bell_sample_metrics.csv"
scan_path = OUTPUT_DIR / "bell_sample_scan.csv"
bell_metrics_df.to_csv(metrics_path, index=False, encoding="utf-8-sig")
data_scan_df.to_csv(scan_path, index=False, encoding="utf-8-sig")

assert set(bell_metrics_df["branch"]) == {"bell_inner_bridge_clip_fill"}
assert set(bell_metrics_df["baseline_source"]) == {"raw_yolo_endocarp"}
assert (bell_metrics_df["top_hull_px"] == 0).all()
assert (bell_metrics_df["top_fill_px"] == 0).all()
assert (bell_metrics_df["final_above_connector_px"] == 0).all()
assert (bell_metrics_df["added_outside_roi_px"] == 0).all()
assert (bell_metrics_df["connector_stem_overlap_px"] == 0).all()
assert (bell_metrics_df["stem_overlap_px"] == 0).all()
assert (bell_metrics_df["components_after"] <= bell_metrics_df["components_before"]).all()
valid = bell_metrics_df[bell_metrics_df["bridge_status"] == "ok"]
assert (valid["endpoints_valid"] == True).all()
assert (valid["left_peak_x"] < valid["stem_left"]).all()
assert (valid["right_peak_x"] > valid["stem_right"]).all()

print(f"Đã test {len(bell_samples)} crop ớt chuông từ {bell_metrics_df['source'].nunique()} ảnh.")
print("Status:", bell_metrics_df["status"].value_counts().to_dict())
print("Tổng pixel thêm:", int(bell_metrics_df["added_px"].sum()))
print("Đã lưu:", figure_path)
print("Đã lưu:", metrics_path)
print(bell_metrics_df.to_string(index=False))

## Kết quả

- [Mở gallery 4 mẫu ớt chuông](runs/endocarp_shape_aware/bell_sample_comparison.png)
- [Mở metrics CSV](runs/endocarp_shape_aware/bell_sample_metrics.csv)
- [Mở log các bbox đã test](runs/endocarp_shape_aware/bell_sample_scan.csv)

Các file kết quả được tham chiếu bằng đường dẫn, không nhúng vào notebook.